# Stage 4/5 — Learn-then-Test 与最终评估（Figure 1）

> 对应 §11、§12。

**λ 不是超参搜索的结果。** 枚举候选阈值挑验证集最优，报告出来的成绩带
后选择偏倚，而 H3 的判据正建立在这个数字上。


In [ ]:
# 让 notebook 能 import sparc（无需 pip install -e .）
import sys, json
from pathlib import Path
CODE_ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))

from sparc.common import load_experiment_config
cfg = load_experiment_config(stage='notebook', run_name='interactive')
print('冻结配置指纹：'); print(json.dumps(cfg.freeze_manifest(), indent=1))


## 1. 固定序列检验的功效诊断

本项目标定折规模在 10² 量级。风险为 0 时，Hoeffding p 值仍需
`n ≥ ln(1/δ)/(2α²) = 150` 才能拒绝 —— 因此从 λ=0.99 起测的固定序列
常在第一步就中断，λ* 恒为 None。这与「真的不存在安全工作点」是两回事。


In [ ]:
import numpy as np
from sparc.calibrate.ltt import diagnose_fixed_sequence_power

gate_demo = np.random.default_rng(0).random(400)   # 换成真实标定折的门控值
d = diagnose_fixed_sequence_power(gate_demo, cfg.prereg.ltt.alpha, cfg.prereg.ltt.delta)
print({k: v for k, v in d.items() if k != 'per_lambda'})
print('冻结配置的多重校正 =', cfg.prereg.ltt.multiple_testing)


## 2. 跑 LTT

```bash
python scripts/run_s4_ltt.py --run-name s4_v1 --losses <calib_losses.npz> --confirm-holdout
```

**跑完之后 S0–S3 被锁死**（§10.4）。要改任何东西只能开新的实验轮次。


In [ ]:
ltt_path = cfg.paths.stage_outputs('s4_ltt') / 's4_v1_ltt.json'
if ltt_path.is_file():
    ltt = json.loads(ltt_path.read_text(encoding='utf-8'))
    print('λ* =', ltt['ltt']['lambda_star'])
    print('覆盖率 =', ltt['ltt']['coverage_at_lambda_star'])
    print(ltt['ltt']['guarantee'])
else:
    print('尚未跑 Stage 4。')


## 3. Figure 1 — Risk–Coverage 曲线

四条曲线：朴素 / Tanimoto 门控 / SPARC-NP / Oracle（不可部署上界）。
`c*` 就是曲线（的 95% UCB）与零伤害线的交点。


In [ ]:
from sparc.eval.safe_coverage import risk_coverage_curve, safe_coverage, oracle_gate_values
from sparc.eval.figures import plot_risk_coverage

# 演示数据；换成 S5 的真实测试集数组
rng = np.random.default_rng(3); n = 800
loss_base = rng.gamma(2.0, 0.5, n)
gate = rng.random(n)
loss_sparc = np.where(gate > 0.5, loss_base - 0.25, loss_base + 0.3)

variants = {'naive': np.ones(n), 'sparc_np': gate, 'oracle': oracle_gate_values(loss_sparc, loss_base)}
curves, c_stars = {}, {}
for name, g in variants.items():
    r = safe_coverage(g, loss_sparc, loss_base, bootstrap_n=2000)
    curves[name], c_stars[name] = r.curve, r.c_star
print(c_stars)
plot_risk_coverage(curves, c_stars)


## 4. Stage 5 最终评估

```bash
python scripts/run_s5_eval.py --run-name s5_final --predictions <test_predictions.npz> \
    --lambda-star <λ*> --deterministic
```

产出 Table 1/2/3、Figure 1/2 与 H1/H2/H3 判定表到 `reports/`。
每张表都强制带均值预测器基线 —— 缺它 `render_table1` 会拒绝渲染（事实 A）。
